# The sunward blind spot

Uses the Python mirror `pymiss` (same formulas and names as `src/physics`). Run from the repo root.


In [ ]:
import sys, math
sys.path.insert(0, '..' if 'notebooks' in __import__('os').getcwd() else '.')
# detection.ts is not mirrored in pymiss; this reproduces its geometric tests with the mirrored photometry.
from pymiss.physics.constants import AU, MU_SUN, DAY
from pymiss.physics.elements import state_to_elements
from pymiss.physics.kepler import Elements, propagate
from pymiss.physics.photometry import absolute_magnitude, apparent_magnitude, view_geometry
home = Elements(AU, 0.0, 0.0, 0.0, 0.0)
T = 365 * DAY
D, p, v_inf = 50.0, 0.14, 15e3
H = absolute_magnitude(D, p)
for come_from in (0, 30, 90, 180):
    hx, hy, hvx, hvy = propagate(home, MU_SUN, T)
    r = math.hypot(hx, hy)
    th = math.radians(come_from)
    sx, sy = -hx / r, -hy / r
    cx, cy = sx * math.cos(th) - sy * math.sin(th), sx * math.sin(th) + sy * math.cos(th)
    rock = state_to_elements(hx, hy, hvx - v_inf * cx, hvy - v_inf * cy, MU_SUN, T, True)
    first = None
    for d in range(365):
        tx, ty, _, _ = propagate(rock, MU_SUN, d * DAY)
        ox, oy, _, _ = propagate(home, MU_SUN, d * DAY)
        r_au, delta_au, alpha, eps = view_geometry(tx, ty, ox, oy)
        if eps >= 60 and apparent_magnitude(H, r_au, delta_au, alpha) <= 24.5:   # Rubin-like ground survey
            first = 365 - d
            break
    print('comes from %3d deg off the sun: ground first sees it %s' % (come_from, '%d days out' % first if first else 'never'))